In [2]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_clean
from src.features import build_dataset

orders, returns = load_clean("../data/raw/online_retail_II.csv")
orders.to_parquet("../data/processed/orders.parquet")
returns.to_parquet("../data/processed/returns.parquet")

df = build_dataset(orders, returns)
df.shape

(77772, 22)

In [4]:
df.groupby("cutoff").agg(n=("customer_id", "size"), base_rate=("target", "mean")).round(3)

,n,base_rate
cutoff,,
2010-06-01,2684,0.286
2010-07-01,2951,0.250
2010-08-01,3136,0.232
2010-09-01,3299,0.272
2010-10-01,3538,0.306
2010-11-01,3913,0.327
2010-12-01,4239,0.191
2011-01-01,4315,0.148
2011-02-01,4387,0.155


In [5]:
assert (df["recency"] >= 1).all()
df["recency"].min()

np.int64(1)

In [6]:
na = df.isna().sum()
na[na > 0]

gap_mean          27341
gap_std           41839
gap_last          27341
recency_vs_gap    27341
dtype: int64

In [7]:
df.sample(10, random_state=42)
df.describe().T

,count,mean,min,25%,50%,75%,max,std
customer_id,77772.0,15343.956977,12346.0,13880.0,15339.5,16823.0,18287.0,1700.570906
frequency,77772.0,4.260737,1.0,1.0,2.0,5.0,222.0,7.09676
monetary_total,77772.0,2175.825451,1.3,302.32,689.16,1754.24,554170.42,10012.978738
aov,77772.0,406.719875,1.3,182.984167,296.1,439.655,39916.5,736.47497
avg_items,77772.0,275.585706,1.0,92.0,159.4,272.0,87167.0,1488.540782
avg_products,77772.0,22.478608,1.0,10.0,18.0,29.0,259.0,18.966856
gap_mean,50431.0,84.46245,1.0,37.0,63.0,108.333333,687.0,72.057454
gap_std,35933.0,46.042259,0.0,18.06227,33.234019,60.775347,440.527525,41.550285
gap_last,50431.0,88.685828,1.0,28.0,61.0,123.0,687.0,85.284271
recency,77772.0,142.585815,1.0,33.0,95.0,215.0,700.0,139.577184


In [8]:
df.to_parquet("../data/processed/dataset.parquet")

## Summary and next steps

- Built a snapshot dataset: 77,772 customer-cutoff rows, 21 feature or helper columns plus the target.
- The target is consistent with the EDA base rates, and the leakage check passed.
- Missing gap features are kept as NaN with a one_order flag.

Next: time-based split (train on earlier cutoffs, validate on later ones, test on the last three), a Logistic Regression baseline, then LightGBM, evaluated with PR-AUC and lift at the top 20%.